In [2]:
%pip install pandas

import pandas as pd



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# World Bank Life Expectancy data
df = pd.read_csv('./API_SP.DYN.LE00.IN_DS2_en_csv_v2_128.csv', skiprows=4)

# Drop the trailing unnamed column World Bank appends
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

df


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Aruba,ABW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,64.049000,64.215000,64.602000,64.944000,65.303000,65.615000,...,75.540000,75.620000,75.880000,76.019000,75.406000,73.655000,76.226000,76.353000,NaN,NaN
1,Africa Eastern and Southern,AFE,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,44.169658,44.468838,44.877890,45.160583,45.535695,45.770723,...,62.167981,62.591275,63.330691,63.857261,63.766484,62.979999,64.487020,65.146154,NaN,NaN
2,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,32.799000,33.291000,33.757000,34.201000,34.673000,35.124000,...,62.646000,62.406000,62.443000,62.941000,61.454000,60.417000,65.617000,66.035000,NaN,NaN
3,Africa Western and Central,AFW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,37.779636,38.058956,38.681792,38.936918,39.194580,39.479784,...,56.392452,56.626439,57.036976,57.149847,57.364425,57.362572,57.987813,58.855722,NaN,NaN
4,Angola,AGO,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,37.933000,36.902000,37.168000,37.419000,37.704000,37.968000,...,61.619000,62.122000,62.622000,63.051000,63.116000,62.958000,64.246000,64.617000,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,Kosovo,XKX,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,48.702000,49.883000,48.378000,50.098000,51.119000,52.003000,...,76.748000,77.105000,77.117000,77.249000,74.212000,74.981000,77.623000,78.033000,NaN,NaN
262,"Yemen, Rep.",YEM,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,33.462000,34.058000,33.669000,33.431000,34.907000,35.592000,...,67.105000,67.120000,65.915000,66.567000,66.435000,66.019000,67.952000,69.295000,NaN,NaN
263,South Africa,ZAF,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,52.575000,53.067000,53.566000,53.895000,54.215000,54.427000,...,64.749000,65.422000,65.726000,66.071000,65.150000,62.010000,65.454000,66.139000,NaN,NaN
264,Zambia,ZMB,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,50.648000,51.041000,51.331000,51.605000,51.170000,52.079000,...,61.129000,61.564000,62.138000,62.914000,63.361000,62.363000,65.279000,66.349000,NaN,NaN


In [4]:
life_expectancy = df.melt(id_vars=['Country Name','Country Code','Indicator Name','Indicator Code'], var_name='Year', value_name='Life expectancy (years)')
life_expectancy['Year'] = pd.to_numeric(life_expectancy['Year'], errors='coerce')
life_expectancy.drop(columns=['Indicator Name', 'Indicator Code'], inplace=True)
life_expectancy

,Country Name,Country Code,Year,Life expectancy (years)
0,Aruba,ABW,1960,64.049000
1,Africa Eastern and Southern,AFE,1960,44.169658
2,Afghanistan,AFG,1960,32.799000
3,Africa Western and Central,AFW,1960,37.779636
4,Angola,AGO,1960,37.933000
...,...,...,...,...
17551,Kosovo,XKX,2025,NaN
17552,"Yemen, Rep.",YEM,2025,NaN
17553,South Africa,ZAF,2025,NaN
17554,Zambia,ZMB,2025,NaN


In [12]:
# Merge country metadata and create grouped DataFrames
meta_country = pd.read_csv('./Metadata_Country_API_SP.DYN.LE00.IN_DS2_en_csv_v2_128.csv')
meta_country
life_expectancy_meta = life_expectancy.merge(meta_country[['Country Code','Region','IncomeGroup','TableName']], on='Country Code', how='left')
life_expectancy_meta.rename(columns={'IncomeGroup':'Income Group'}, inplace=True)
life_expectancy_meta.drop(columns=['TableName'], inplace=True)
life_expectancy_meta

,Country Name,Country Code,Year,Life expectancy (years),Region,Income Group
0,Aruba,ABW,1960,64.049000,Latin America & Caribbean,High income
1,Africa Eastern and Southern,AFE,1960,44.169658,NaN,NaN
2,Afghanistan,AFG,1960,32.799000,Middle East & North Africa,Low income
3,Africa Western and Central,AFW,1960,37.779636,NaN,NaN
4,Angola,AGO,1960,37.933000,Sub-Saharan Africa,Lower middle income
...,...,...,...,...,...,...
17551,Kosovo,XKX,2025,NaN,Europe & Central Asia,Upper middle income
17552,"Yemen, Rep.",YEM,2025,NaN,Middle East & North Africa,Low income
17553,South Africa,ZAF,2025,NaN,Sub-Saharan Africa,Upper middle income
17554,Zambia,ZMB,2025,NaN,Sub-Saharan Africa,Lower middle income


In [ ]:
# By-region time series (mean of life expectancy per year per region)

# Use the long-format dataframe for Tableau or Plotly
by_region = life_expectancy_meta.groupby(['Region','Year'])['Life expectancy (years)'].mean().reset_index()

# Output a CSV for Tableau
by_region.to_csv('life_expectancy_by_region.csv', index=False)

# Use the wide-format dataframe for Matplotlib
by_region_pivot = by_region.pivot(index='Year', columns='Region', values='Life expectancy (years)')

by_region


,Region,Year,Life expectancy (years)
0,East Asia & Pacific,1960,54.503296
1,East Asia & Pacific,1961,55.157497
2,East Asia & Pacific,1962,55.876325
3,East Asia & Pacific,1963,56.343065
4,East Asia & Pacific,1964,56.721374
...,...,...,...
457,Sub-Saharan Africa,2021,62.177204
458,Sub-Saharan Africa,2022,63.060397
459,Sub-Saharan Africa,2023,64.547890
460,Sub-Saharan Africa,2024,NaN


In [17]:
# By-income-group time series
by_income = life_expectancy_meta.groupby(['Income Group','Year'])['Life expectancy (years)'].mean().reset_index()
by_income.to_csv('life_expectancy_by_income_group.csv', index=False)

by_income_pivot = by_income.pivot(index='Year', columns='Income Group', values='Life expectancy (years)')
by_income_pivot.head()


Income Group,High income,Low income,Lower middle income,Upper middle income
Year,,,,
1960,65.075805,39.91784,45.943889,54.181275
1961,65.515986,40.24396,46.434105,54.822414
1962,65.704132,40.56628,47.087975,55.537212
1963,65.936763,40.60052,47.479726,56.190231
1964,66.376732,41.21896,48.017306,56.698950


In [15]:
# If the region is missing, it's not a country, so we can drop those entries for the country-level dataset
by_country = life_expectancy_meta.dropna(subset=['Region']).reset_index(drop=True)
by_country.to_csv('life_expectancy_by_country.csv', index=False)
by_country

,Country Name,Country Code,Year,Life expectancy (years),Region,Income Group
0,Aruba,ABW,1960,64.049,Latin America & Caribbean,High income
1,Afghanistan,AFG,1960,32.799,Middle East & North Africa,Low income
2,Angola,AGO,1960,37.933,Sub-Saharan Africa,Lower middle income
3,Albania,ALB,1960,56.413,Europe & Central Asia,Upper middle income
4,Andorra,AND,1960,72.094,Europe & Central Asia,High income
...,...,...,...,...,...,...
14317,Kosovo,XKX,2025,NaN,Europe & Central Asia,Upper middle income
14318,"Yemen, Rep.",YEM,2025,NaN,Middle East & North Africa,Low income
14319,South Africa,ZAF,2025,NaN,Sub-Saharan Africa,Upper middle income
14320,Zambia,ZMB,2025,NaN,Sub-Saharan Africa,Lower middle income
